In [1]:
import os
import numpy as np
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16

# ==========================================
# 0. กำหนด Path ของ Dataset
# ==========================================
base_dir = './cats_and_dogs_small'
train_dir = os.path.join(base_dir, 'train')
validation_dir = os.path.join(base_dir, 'validation')
test_dir = os.path.join(base_dir, 'test')

# ==========================================
# วิธีที่ 1: เทรน CNN ขึ้นมาใหม่เองทั้งหมด + Data Augmentation
# ==========================================
model_1 = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(512, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model_1.compile(loss='binary_crossentropy',
                optimizer=optimizers.RMSprop(learning_rate=1e-4),
                metrics=['accuracy'])

train_datagen_aug = ImageDataGenerator(
    rescale=1./255, rotation_range=40, width_shift_range=0.2,
    height_shift_range=0.2, shear_range=0.2, zoom_range=0.2,
    horizontal_flip=True, fill_mode='nearest')

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator_1 = train_datagen_aug.flow_from_directory(
    train_dir, target_size=(150, 150), batch_size=32, class_mode='binary')

validation_generator_1 = test_datagen.flow_from_directory(
    validation_dir, target_size=(150, 150), batch_size=32, class_mode='binary')

print("--- เริ่มเทรนวิธีที่ 1: CNN จากศูนย์ ---")
history_1 = model_1.fit(
    train_generator_1, steps_per_epoch=100, epochs=100,
    validation_data=validation_generator_1, validation_steps=50)

model_1.save('cats_and_dogs_small_2.h5')


# ==========================================
# วิธีที่ 2: VGG16 สกัด Feature ล่วงหน้าอย่างรวดเร็ว (ไม่ทำ Augmentation)
# ==========================================
conv_base = VGG16(weights='imagenet', include_top=False, input_shape=(150, 150, 3))
datagen_basic = ImageDataGenerator(rescale=1./255)
batch_size = 20

def extract_features(directory, sample_count):
    features = np.zeros(shape=(sample_count, 4, 4, 512))
    labels = np.zeros(shape=(sample_count))
    generator = datagen_basic.flow_from_directory(
        directory, target_size=(150, 150), batch_size=batch_size, class_mode='binary')
    i = 0
    for inputs_batch, labels_batch in generator:
        features_batch = conv_base.predict(inputs_batch)
        current_batch_size = len(features_batch)
        features[i * batch_size : i * batch_size + current_batch_size] = features_batch
        labels[i * batch_size : i * batch_size + current_batch_size] = labels_batch
        i += 1
        if i * batch_size >= sample_count:
            break
    return features, labels

train_features, train_labels = extract_features(train_dir, 2000)
validation_features, validation_labels = extract_features(validation_dir, 1000)

train_features = np.reshape(train_features, (2000, 4 * 4 * 512))
validation_features = np.reshape(validation_features, (1000, 4 * 4 * 512))

model_2 = models.Sequential([
    layers.Dense(256, activation='relu', input_dim=4 * 4 * 512),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model_2.compile(optimizer=optimizers.RMSprop(learning_rate=2e-5),
                loss='binary_crossentropy', metrics=['accuracy'])

print("--- เริ่มเทรนวิธีที่ 2: VGG16 Feature Extraction ---")
history_2 = model_2.fit(train_features, train_labels,
                        epochs=30, batch_size=20,
                        validation_data=(validation_features, validation_labels))


# ==========================================
# วิธีที่ 3: VGG16 แบบประกอบร่างเข้าด้วยกัน + Data Augmentation (Freeze Base)
# ==========================================
model_3 = models.Sequential([
    conv_base,
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

# สั่งห้ามเทรน Weight ในส่วนของ VGG16 ป้องกันไม่ให้ Weight พัง
conv_base.trainable = False 

model_3.compile(loss='binary_crossentropy',
                optimizer=optimizers.RMSprop(learning_rate=2e-5),
                metrics=['accuracy'])

# สร้าง Generator ใหม่เพราะ Batch Size ในโค้ดเดิมตรงนี้ถูกปรับเป็น 20
train_generator_3 = train_datagen_aug.flow_from_directory(
    train_dir, target_size=(150, 150), batch_size=20, class_mode='binary')

validation_generator_3 = test_datagen.flow_from_directory(
    validation_dir, target_size=(150, 150), batch_size=20, class_mode='binary')

print("--- เริ่มเทรนวิธีที่ 3: VGG16 + Augmentation ---")
history_3 = model_3.fit(
    train_generator_3, steps_per_epoch=100, epochs=30,
    validation_data=validation_generator_3, validation_steps=50)

/Users/ponlawatchangto/Documents/Mylearning/4312/.venv/lib/python3.13/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


Found 1998 images belonging to 2 classes.
Found 1000 images belonging to 2 classes.


/Users/ponlawatchangto/Documents/Mylearning/4312/.venv/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


--- เริ่มเทรนวิธีที่ 1: CNN จากศูนย์ ---
Epoch 1/100
 54/100 ━━━━━━━━━━━━━━━━━━━━ 4s 102ms/step - accuracy: 0.4839 - loss: 0.6964

/Users/ponlawatchangto/Documents/Mylearning/4312/.venv/lib/python3.13/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


...

100/100 ━━━━━━━━━━━━━━━━━━━━ 8s 74ms/step - accuracy: 0.4950 - loss: 0.6950 - val_accuracy: 0.5060 - val_loss: 0.6904
Epoch 2/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 7s 73ms/step - accuracy: 0.5350 - loss: 0.6910 - val_accuracy: 0.5680 - val_loss: 0.6855
Epoch 3/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 7s 73ms/step - accuracy: 0.5666 - loss: 0.6828 - val_accuracy: 0.5660 - val_loss: 0.6796
Epoch 4/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 7s 73ms/step - accuracy: 0.5626 - loss: 0.6806 - val_accuracy: 0.5590 - val_loss: 0.6792
Epoch 5/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 7s 73ms/step - accuracy: 0.5916 - loss: 0.6704 - val_accuracy: 0.5860 - val_loss: 0.6690
Epoch 6/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 7s 73ms/step - accuracy: 0.5901 - loss: 0.6686 - val_accuracy: 0.5940 - val_loss: 0.6638
Epoch 7/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 7s 74ms/step - accuracy: 0.6001 - loss: 0.6638 - val_accuracy: 0.6090 - val_loss: 0.6578
Epoch 8/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 8s 76ms/step - accuracy: 0.6036 - loss: 0.6564 - val_accuracy

Found 1998 images belonging to 2 classes.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 405ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 360ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 360ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 362ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 381ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 363ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 360ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 366ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 360ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 369ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 361ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 362ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 360ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 363ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 362ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 381ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 366ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 362ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 363ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 362ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 381ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 361ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 362ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 361ms/step
1/1 ━━━━━━━━━━━━━━━━━━

/Users/ponlawatchangto/Documents/Mylearning/4312/.venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6715 - loss: 0.6053 - val_accuracy: 0.7860 - val_loss: 0.4669
Epoch 2/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8085 - loss: 0.4263 - val_accuracy: 0.8460 - val_loss: 0.3737
Epoch 3/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8465 - loss: 0.3621 - val_accuracy: 0.8560 - val_loss: 0.3409
Epoch 4/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8735 - loss: 0.3189 - val_accuracy: 0.8690 - val_loss: 0.3179
Epoch 5/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8750 - loss: 0.2997 - val_accuracy: 0.8780 - val_loss: 0.3001
Epoch 6/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8850 - loss: 0.2747 - val_accuracy: 0.8760 - val_loss: 0.2941
Epoch 7/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8995 - loss: 0.2655 - val_accuracy: 0.8820 - val_loss: 0.2822
Epoch 8/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9110 - loss: 0.2453 - val_accuracy: 0.8880 - val_

KeyboardInterrupt: 